# 🤖 Notebook 4 — Model Training

**Project:** Customer Churn Prediction  
**Objective:** Encode features, split the data, train multiple ML models, and compare performance.

**Models trained:**
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost (if installed)

**Selection criterion:** Highest ROC-AUC score.

---

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sys.path.insert(0, os.path.join('..', 'src'))
from utils import load_dataset, evaluate_model, print_metrics_table, save_model, ensure_dir
from preprocessing import run_preprocessing_pipeline

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print('XGBoost available ✅')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — will skip XGBClassifier.')

sns.set_theme(style='whitegrid')
PLOTS_DIR  = os.path.join('..', 'outputs', 'plots')
MODEL_PATH = os.path.join('..', 'model', 'churn_model.pkl')
ensure_dir(PLOTS_DIR)
print('Libraries loaded ✅')

## 4.1  Load & Preprocess Data

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'customer_churn.csv')
df = load_dataset(DATA_PATH)

# Full preprocessing pipeline: clean → feature engineer → encode → split → scale
X_train, X_test, y_train, y_test, scaler, feature_names, encoding_info = \
    run_preprocessing_pipeline(df)

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train Churn rate: {y_train.mean():.3f}')
print(f'y_test  Churn rate: {y_test.mean():.3f}')

## 4.2  Define Models

In [ ]:
# Each model uses class_weight='balanced' to handle class imbalance
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced', solver='lbfgs'
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=10, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=5,
        random_state=42, class_weight='balanced', n_jobs=-1
    ),
}

if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=3, random_state=42,
        eval_metric='logloss', verbosity=0
    )

print(f'Models to train: {list(models.keys())}')

## 4.3  Train All Models

In [ ]:
trained_models = {}
all_metrics    = []

for name, model in models.items():
    print(f'\nTraining: {name} ...')
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test, model_name=name)
    trained_models[name] = model
    all_metrics.append(metrics)
    print(f'  ✅  Done — ROC-AUC: {metrics["roc_auc"]}')

## 4.4  Performance Comparison Table

In [ ]:
print_metrics_table(all_metrics)

# Display as styled DataFrame
metrics_df = pd.DataFrame(all_metrics).set_index('model')
metrics_df.style.highlight_max(axis=0, color='lightgreen')

## 4.5  Metric Visualisation

In [ ]:
metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
plot_df = pd.DataFrame(all_metrics).set_index('model')[metric_cols]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(plot_df.index))
width = 0.15
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, (col, color) in enumerate(zip(metric_cols, colors)):
    bars = ax.bar(x + i * width, plot_df[col], width, label=col.upper(), color=color, alpha=0.85)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(plot_df.index, fontsize=10)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '11_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4.6  Select Best Model (by ROC-AUC)

In [ ]:
valid_metrics = [m for m in all_metrics if isinstance(m['roc_auc'], float)]
best_metrics  = max(valid_metrics, key=lambda m: m['roc_auc'])
best_name     = best_metrics['model']
best_model    = trained_models[best_name]

print(f'\n🏆  Best Model : {best_name}')
print(f'   Accuracy   : {best_metrics["accuracy"]}')
print(f'   Precision  : {best_metrics["precision"]}')
print(f'   Recall     : {best_metrics["recall"]}')
print(f'   F1 Score   : {best_metrics["f1"]}')
print(f'   ROC-AUC    : {best_metrics["roc_auc"]}')

## 4.7  Save Best Model

In [ ]:
# Bundle model with its metadata for use in predict.py
model_bundle = {
    'model'        : best_model,
    'model_name'   : best_name,
    'scaler'       : scaler,
    'feature_names': feature_names,
    'encoding_info': encoding_info,
}

save_model(model_bundle, MODEL_PATH)
print(f'\n✅  Model bundle saved to: {MODEL_PATH}')

---
## Summary

All models trained and compared. The best model (by ROC-AUC) has been saved to `model/churn_model.pkl`.

➡  **Next:** `05_model_evaluation.ipynb` — deep-dive evaluation: confusion matrix, ROC curve, feature importance.